# Parameter Sensitivity Analysis (One-at-a-Time)

This notebook analyzes how `Urc1_Adaptive` hyperparameters affect model behavior and GT alignment quality.

## Experimental Design
- Method: one-at-a-time sensitivity analysis
- Principle: change one parameter while keeping all others fixed
- Target parameters:
  1. `initial_window_size`
  2. `fixed_freshness_threshold`
  3. `decay_rate`

## Core Metrics
- `Data Points (n)` (higher is better)
- `RMSE (mV)` (lower is better)
- `Monotonicity` (higher is better)
- `Slope Sigma (uV/h)` (lower is better)
- `Outlier (%)` (lower is better)

## GT Metrics
- `GT RMSE (mV)` (lower is better)
- `GT MAE (mV)` (lower is better)
- `GT Valid Intervals` (higher is better)

In [ ]:
# =============================================================================
# Cell 1: Imports
# =============================================================================
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
from IPython.display import display
from plotly.subplots import make_subplots
from tqdm.notebook import tqdm

from degradation_toolbox.Urc.Urc1 import Urc1
from degradation_toolbox.Urc.Urc1_adaptive import Urc1_Adaptive
from master_arbeit_Di.explore.GMpreprocess import GMpreprocess
from master_arbeit_Di.explore.UnifiedModelComparator import UnifiedModelComparator
from master_arbeit_Di.utils.stability_evaluation import evaluate_urc_stability

warnings.filterwarnings('ignore')
print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
# =============================================================================
# Cell 3: Configuration
# =============================================================================

# Dataset and directories
DATASET_PATH = r"..\\..\\explore_data\\G6M2.parquet"
PREPROCESS_OUTPUT_DIR = r"..\\..\\explore_data\\output"
PLOTS_OUTPUT_DIR = Path("..\\plots\\adaptive\\G6M2_sensitivity")
SAVE_PLOTS = True
PLOTS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Reference condition configurations: Low, Medium, High
REF_CONFIGS = {
    "Low": {
        "Iref": 0.28,
        "Tref": 57,
        "OHref": 10,
        "gt_file": r"..\\ground_truth\\output_backup\\gt_raw_processed\\G6M2__gt_diagram__g6m2_low_load__iref_0p28__tref_57__ohref_10__daily_regression_full_coverage.csv",
    },
    "Medium": {
        "Iref": 1.0,
        "Tref": 58,
        "OHref": 33,
        "gt_file": r"..\\ground_truth\\output_backup\\gt_raw_processed\\G6M2__gt_diagram__g6m2_mid_load__iref_1__tref_58__ohref_33__daily_regression_full_coverage.csv",
    },
    "High": {
        "Iref": 1.31,
        "Tref": 58,
        "OHref": 18,
        "gt_file": r"..\\ground_truth\\output_backup\\gt_raw_processed\\G6M2__gt_diagram__g6m2_high_load__iref_1p31__tref_58__ohref_18__daily_regression_full_coverage.csv",
    },
}

# Derived reference current list (for evaluation loops)
IREF_LIST = [cfg["Iref"] for cfg in REF_CONFIGS.values()]

# Shared model config
COMMON_CONFIG = {
    "Iref": IREF_LIST,
    "Tref": 60,
    "OHref": 72,
    "ref_config": REF_CONFIGS,
    "len_interval": 1,
    "slide": 1,
    "min_num_data_required_for_fit": 300,
    "threshold": 1e6,
    "i_off": 0.1,
    "u_off": 1.3,
    "plot_fit": 0,
    "data_filter_i_min": 0.1,
    "data_filter_U_min": 1.4,
    "data_filter_U_max": 2.3,
    "data_filter_T_min": 50,
    "data_filter_T_max": 65,
}

# Adaptive default parameters (baseline for sensitivity analysis)
ADAPTIVE_DEFAULT = {
    "initial_window_size": 3,
    "min_target_count_ratio": 0.1,
    "decay_rate": 0.2,
    "urc_min": 1.4,
    "urc_max": 2.4,
    "fixed_freshness_threshold": 0.3,
}

print("=" * 70)
print("Reference conditions")
print("=" * 70)
for name, cfg in REF_CONFIGS.items():
    print(f"  {name}: Iref={cfg['Iref']}, Tref={cfg['Tref']}, OHref={cfg['OHref']}")
print("\nAdaptive defaults")
for k, v in ADAPTIVE_DEFAULT.items():
    print(f"  {k}: {v}")
print(f"\nPlot output dir: {PLOTS_OUTPUT_DIR.resolve()}")

Reference conditions
  Low: Iref=0.28, Tref=57, OHref=10
  Medium: Iref=1.0, Tref=58, OHref=33
  High: Iref=1.31, Tref=58, OHref=18

Adaptive defaults
  initial_window_size: 3
  min_target_count_ratio: 0.1
  decay_rate: 0.2
  urc_min: 1.4
  urc_max: 2.4
  fixed_freshness_threshold: 0.3

Plot output dir: C:\Users\Z0057NPT\Documents\MA_code\master_arbeit_Di\plots\ada\G6M2_sensitivity


In [3]:
# =============================================================================
# Cell 2: Data Loading & Preprocessing
# =============================================================================

preprocessor = GMpreprocess(file_path=DATASET_PATH, output_dir=PREPROCESS_OUTPUT_DIR)
data = preprocessor.run()
extracted_name = preprocessor.name

if data is None:
    raise RuntimeError(
        "GMpreprocess.run() returned None. Check input file path and raw columns."
    )

print(f"Dataset: {extracted_name}")
print(f"Data shape: {data.shape}")

# Load GT series for each reference current (used by comparator GT metrics)
GT_BY_IREF = {}
for ref_name, ref_cfg in REF_CONFIGS.items():
    gt_path = Path(ref_cfg["gt_file"])
    if not gt_path.is_absolute():
        gt_path = (Path.cwd() / gt_path).resolve()

    if not gt_path.exists():
        print(f"[WARN] GT file missing for {ref_name}: {gt_path}")
        continue

    try:
        gt_df = pd.read_csv(gt_path, index_col=0, parse_dates=True)
        colmap = {str(c).strip().lower(): c for c in gt_df.columns}
        candidate_cols = ["gt_uref_regression", "voltage", "uref", "gt_uref"]
        selected_col = next((colmap[c] for c in candidate_cols if c in colmap), None)

        if selected_col is None:
            if len(gt_df.columns) == 1:
                selected_col = gt_df.columns[0]
            else:
                raise ValueError(f"Cannot infer GT voltage column from: {gt_df.columns.tolist()}")

        gt_series = gt_df[selected_col].dropna()
        GT_BY_IREF[ref_cfg["Iref"]] = gt_series
        print(f"[OK] {ref_name} GT loaded: Iref={ref_cfg['Iref']}, n={len(gt_series)}, col={selected_col}")

    except Exception as e:
        print(f"[WARN] Failed loading GT for {ref_name}: {e}")

print(f"\nGT references available: {len(GT_BY_IREF)}/{len(REF_CONFIGS)}")

=== 1. Loading & Preprocessing: G6M2 ===
>> Data loaded successfully.
   [Detected] Temperature Col: 'Temp_Module_12' -> ID: '2'
>> Columns renamed to standard format.
>> Columns filtered. Retained: ['currentDensity', 'temperature', 'voltage']

=== 3. Saving Preprocessd data in .parquet format ===
>> ✅ Final Results saved successfully to:
   ..\\..\\explore_data\\output\G6M2_20260502_120213.parquet

=== GMpreprocess Pipeline Completed Successfully ===
Dataset: G6M2
Data shape: (1096020, 3)
[OK] Low GT loaded: Iref=0.28, n=758, col=gt_uref_regression
[OK] Medium GT loaded: Iref=1.0, n=758, col=gt_uref_regression
[OK] High GT loaded: Iref=1.31, n=758, col=gt_uref_regression

GT references available: 3/3


In [4]:
# =============================================================================
# Cell 4: Parameter search space
# =============================================================================

PARAM_GRIDS = {
    "initial_window_size": [2, 3, 4, 5, 6],
    "fixed_freshness_threshold": [0.2, 0.25, 0.3, 0.35, 0.4],
    "decay_rate": [0.1, 0.15, 0.2, 0.25, 0.3],
}

print("Parameter grids:")
for param, values in PARAM_GRIDS.items():
    print(f"  {param}: {values}")

Parameter grids:
  initial_window_size: [2, 3, 4, 5, 6]
  fixed_freshness_threshold: [0.2, 0.25, 0.3, 0.35, 0.4]
  decay_rate: [0.1, 0.15, 0.2, 0.25, 0.3]


In [5]:
# =============================================================================
# Cell 5: Train baseline model (reference)
# =============================================================================
print("Training baseline model...")

shared_pre = Urc1.preprocess_once(
    data,
    i_off=COMMON_CONFIG["i_off"],
    u_off=COMMON_CONFIG["u_off"],
    data_filter_i_min=COMMON_CONFIG["data_filter_i_min"],
    data_filter_U_min=COMMON_CONFIG["data_filter_U_min"],
    data_filter_U_max=COMMON_CONFIG["data_filter_U_max"],
    data_filter_T_min=COMMON_CONFIG["data_filter_T_min"],
    data_filter_T_max=COMMON_CONFIG["data_filter_T_max"],
)
print(f"Shared preprocessed rows: {len(shared_pre)}")

urc_baseline = Urc1(
    data=data,
    name=extracted_name,
    preprocessed_data=shared_pre,
    **COMMON_CONFIG,
)

# Baseline non-GT metrics
baseline_metrics = {}
for i_ref in IREF_LIST:
    m = evaluate_urc_stability(urc_baseline, i_ref)
    if m:
        baseline_metrics[i_ref] = m

# Baseline GT metrics using UnifiedModelComparator
baseline_gt_metrics = {}
baseline_cmp = UnifiedModelComparator({"Baseline": urc_baseline})
for i_ref, gt_series in GT_BY_IREF.items():
    baseline_cmp.set_ground_truth(gt_series, iref=i_ref)

for i_ref in IREF_LIST:
    try:
        df_gt = baseline_cmp.compare_all(
            i_target=i_ref,
            include_gt_metrics=True,
            include_all_cond_metrics=False,
        )
        if not df_gt.empty:
            row = df_gt.iloc[0].to_dict()
            baseline_gt_metrics[i_ref] = {
                "GT RMSE (mV)": row.get("GT RMSE (mV)", np.nan),
                "GT MAE (mV)": row.get("GT MAE (mV)", np.nan),
                "GT Valid Intervals": row.get("GT Valid Intervals", np.nan),
            }
    except Exception:
        baseline_gt_metrics[i_ref] = {
            "GT RMSE (mV)": np.nan,
            "GT MAE (mV)": np.nan,
            "GT Valid Intervals": np.nan,
        }

print("\nBaseline stability metrics:")
display(pd.DataFrame(baseline_metrics).T)

print("Baseline GT metrics:")
display(pd.DataFrame(baseline_gt_metrics).T)

Training baseline model...
[preprocess_once] 1096020 -> 359353 points.
Shared preprocessed rows: 359353
Using shared preprocessed data (359353 points, skipping preprocess).
Voltage model fitting ...
Fitting Stats: 323 intervals low data, 0 fit failed.
350 out of 762 fitting results are reliable.

Baseline stability metrics:


,Target Current (A/cm2),Data Points (n),RMSE (mV),Slope Sigma (uV/h),Mono (Rank) [0-1],Outlier Count,Outlier (%),Max Residual (mV),Mean SE (mV)
0.28,0.28,350.0,7.677,0.079,0.907,6.0,1.71,87.016,1.687
1.00,1.00,350.0,8.044,0.083,0.972,4.0,1.14,52.413,1.710
1.31,1.31,350.0,12.103,0.125,0.971,3.0,0.86,71.629,2.250


Baseline GT metrics:


,GT RMSE (mV),GT MAE (mV),GT Valid Intervals
0.28,10.043,7.482,350.0
1.00,15.586,13.537,350.0
1.31,15.687,11.931,350.0


In [6]:
# =============================================================================
# Cell 6: Define experiment runner function
# =============================================================================

def run_sensitivity_experiment(
    param_name: str,
    param_values: list,
    data,
    name,
    shared_pre,
    common_config,
    adaptive_default,
    iref_list,
    gt_by_iref,
):
    """Run one-at-a-time sensitivity experiments for a single parameter.

    Returns a dataframe containing both stability and GT metrics.
    """
    results = []

    print(f"\n{'=' * 70}")
    print(f"Testing parameter: {param_name}")
    print(f"Values: {param_values}")
    print(f"{'=' * 70}")

    for value in tqdm(param_values, desc=f"Testing {param_name}"):
        ada_cfg = adaptive_default.copy()
        ada_cfg[param_name] = value

        try:
            urc_model = Urc1_Adaptive(
                data=data,
                name=name,
                preprocessed_data=shared_pre,
                **common_config,
                **ada_cfg,
            )

            cmp = UnifiedModelComparator({"Adaptive": urc_model})
            for i_ref, gt_series in gt_by_iref.items():
                cmp.set_ground_truth(gt_series, iref=i_ref)

            for i_ref in iref_list:
                stability = evaluate_urc_stability(urc_model, i_ref) or {}

                gt_rmse = np.nan
                gt_mae = np.nan
                gt_valid = np.nan
                try:
                    df_gt = cmp.compare_all(
                        i_target=i_ref,
                        include_gt_metrics=True,
                        include_all_cond_metrics=False,
                    )
                    if not df_gt.empty:
                        row_gt = df_gt.iloc[0]
                        gt_rmse = row_gt.get("GT RMSE (mV)", np.nan)
                        gt_mae = row_gt.get("GT MAE (mV)", np.nan)
                        gt_valid = row_gt.get("GT Valid Intervals", np.nan)
                except Exception:
                    pass

                row = dict(stability)
                row["GT RMSE (mV)"] = gt_rmse
                row["GT MAE (mV)"] = gt_mae
                row["GT Valid Intervals"] = gt_valid
                row["param_name"] = param_name
                row["param_value"] = value
                row["i_ref"] = i_ref
                results.append(row)

        except Exception as e:
            print(f"  Warning: {param_name}={value} failed: {e}")
            continue

    return pd.DataFrame(results)

print("Experiment function defined.")

Experiment function defined.


---
# Experiment 1: `initial_window_size`

`initial_window_size` controls the minimum number of days used before adaptive expansion begins.

In [7]:
# =============================================================================
# Cell 7: Run initial_window_size experiment
# =============================================================================

results_window_size = run_sensitivity_experiment(
    param_name="initial_window_size",
    param_values=PARAM_GRIDS["initial_window_size"],
    data=data,
    name=extracted_name,
    shared_pre=shared_pre,
    common_config=COMMON_CONFIG,
    adaptive_default=ADAPTIVE_DEFAULT,
    iref_list=IREF_LIST,
    gt_by_iref=GT_BY_IREF,
)

print(f"\nCollected {len(results_window_size)} rows")
display(results_window_size.head(10))


Testing parameter: initial_window_size
Values: [2, 3, 4, 5, 6]


Testing initial_window_size:   0%|          | 0/5 [00:00<?, ?it/s]

Using shared preprocessed data (359353 points, skipping preprocess).
Voltage model fitting ...
Starting Adaptive Fitting for approx 757 intervals...
400 out of 762 fitting results are reliable.
Initialized Urc1_Adaptive (Simplified): Init_Win=2d
  Filters: Target_Min_Ratio=0.1, Freshness=0.3
Using shared preprocessed data (359353 points, skipping preprocess).
Voltage model fitting ...
Starting Adaptive Fitting for approx 757 intervals...
315 out of 762 fitting results are reliable.
Initialized Urc1_Adaptive (Simplified): Init_Win=3d
  Filters: Target_Min_Ratio=0.1, Freshness=0.3
Using shared preprocessed data (359353 points, skipping preprocess).
Voltage model fitting ...
Starting Adaptive Fitting for approx 757 intervals...
224 out of 762 fitting results are reliable.
Initialized Urc1_Adaptive (Simplified): Init_Win=4d
  Filters: Target_Min_Ratio=0.1, Freshness=0.3
Using shared preprocessed data (359353 points, skipping preprocess).
Voltage model fitting ...
Starting Adaptive Fitting 

,Target Current (A/cm2),Data Points (n),RMSE (mV),Slope Sigma (uV/h),Mono (Rank) [0-1],Outlier Count,Outlier (%),Max Residual (mV),Mean SE (mV),GT RMSE (mV),GT MAE (mV),GT Valid Intervals,param_name,param_value,i_ref
0,0.28,400,3.049,0.030,0.979,7,1.75,13.271,1.794,7.939,7.167,400,initial_window_size,2,0.28
1,1.00,400,6.964,0.069,0.982,1,0.25,21.413,1.854,14.404,12.800,400,initial_window_size,2,1.00
2,1.31,400,10.333,0.102,0.982,2,0.50,44.298,2.268,13.894,10.966,400,initial_window_size,2,1.31
3,0.28,315,2.952,0.033,0.982,4,1.27,13.313,1.898,8.229,7.462,315,initial_window_size,3,0.28
4,1.00,315,6.756,0.076,0.984,0,0.00,17.974,1.937,14.057,12.478,315,initial_window_size,3,1.00
5,1.31,315,9.812,0.110,0.984,0,0.00,26.385,2.255,13.124,10.298,315,initial_window_size,3,1.31
6,0.28,224,2.868,0.038,0.985,4,1.79,12.432,2.072,8.751,7.985,224,initial_window_size,4,0.28
7,1.00,224,6.520,0.087,0.981,0,0.00,17.832,2.113,13.515,12.040,224,initial_window_size,4,1.00
8,1.31,224,9.167,0.122,0.983,0,0.00,25.627,2.442,11.632,9.020,224,initial_window_size,4,1.31
9,0.28,169,2.500,0.039,0.989,1,0.59,7.545,2.115,8.524,7.914,169,initial_window_size,5,0.28


In [8]:
# =============================================================================
# Cell 8: Plot helper + initial_window_size plots
# =============================================================================

def plot_sensitivity_results(df, param_name, baseline_metrics, title_suffix=""):
    """Plot one-parameter sensitivity results for core stability metrics."""
    metrics_to_plot = [
        ("RMSE (mV)", "RMSE", "lower"),
        ("Slope Sigma (uV/h)", "Slope Uncertainty", "lower"),
        ("Outlier (%)", "Outlier %", "lower"),
        ("Mono (Rank) [0-1]", "Monotonicity", "higher"),
        ("Data Points (n)", "Data Points", "higher"),
    ]

    fig = make_subplots(
        rows=2,
        cols=3,
        subplot_titles=[m[1] for m in metrics_to_plot] + [""],
        vertical_spacing=0.15,
        horizontal_spacing=0.08,
    )

    irefs = sorted(df["i_ref"].dropna().unique().tolist())
    color_seq = ["#636EFA", "#EF553B", "#00CC96", "#AB63FA", "#FFA15A"]
    colors = {str(i): color_seq[idx % len(color_seq)] for idx, i in enumerate(irefs)}

    for idx, (metric_col, metric_name, better) in enumerate(metrics_to_plot):
        row = idx // 3 + 1
        col = idx % 3 + 1

        for i_ref in irefs:
            df_i = df[df["i_ref"] == i_ref]
            if df_i.empty or metric_col not in df_i.columns:
                continue

            fig.add_trace(
                go.Scatter(
                    x=df_i["param_value"],
                    y=df_i[metric_col],
                    mode="lines+markers",
                    name=f"{i_ref} A/cm²",
                    line=dict(color=colors[str(i_ref)]),
                    legendgroup=str(i_ref),
                    showlegend=(idx == 0),
                ),
                row=row,
                col=col,
            )

            if i_ref in baseline_metrics and metric_col in baseline_metrics[i_ref]:
                baseline_val = baseline_metrics[i_ref][metric_col]
                fig.add_hline(
                    y=baseline_val,
                    line_dash="dash",
                    line_color=colors[str(i_ref)],
                    opacity=0.5,
                    row=row,
                    col=col,
                )

        arrow = "↑" if better == "higher" else "↓"
        fig.update_yaxes(title_text=f"{metric_name} ({arrow} better)", row=row, col=col)

    fig.update_layout(
        height=640,
        title_text=f"Sensitivity Analysis: {param_name} {title_suffix}",
        template="plotly_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    )

    for c in [1, 2, 3]:
        fig.update_xaxes(title_text=param_name, row=2, col=c)

    fig.show()
    return fig


def plot_gt_sensitivity(df, param_name, baseline_gt_metrics, title_suffix=""):
    """Plot GT alignment metrics across parameter values."""
    gt_metrics = [
        ("GT RMSE (mV)", "GT RMSE", "lower"),
        ("GT MAE (mV)", "GT MAE", "lower"),
        ("GT Valid Intervals", "GT Valid Intervals", "higher"),
    ]

    fig = make_subplots(
        rows=1,
        cols=3,
        subplot_titles=[m[1] for m in gt_metrics],
        horizontal_spacing=0.08,
    )

    irefs = sorted(df["i_ref"].dropna().unique().tolist())
    color_seq = ["#636EFA", "#EF553B", "#00CC96", "#AB63FA", "#FFA15A"]
    colors = {str(i): color_seq[idx % len(color_seq)] for idx, i in enumerate(irefs)}

    for idx, (metric_col, metric_name, better) in enumerate(gt_metrics):
        col = idx + 1
        for i_ref in irefs:
            df_i = df[df["i_ref"] == i_ref]
            if df_i.empty or metric_col not in df_i.columns:
                continue

            fig.add_trace(
                go.Scatter(
                    x=df_i["param_value"],
                    y=df_i[metric_col],
                    mode="lines+markers",
                    name=f"{i_ref} A/cm²",
                    line=dict(color=colors[str(i_ref)]),
                    legendgroup=f"gt_{i_ref}",
                    showlegend=(idx == 0),
                ),
                row=1,
                col=col,
            )

            if i_ref in baseline_gt_metrics and metric_col in baseline_gt_metrics[i_ref]:
                baseline_val = baseline_gt_metrics[i_ref][metric_col]
                if pd.notna(baseline_val):
                    fig.add_hline(
                        y=baseline_val,
                        line_dash="dash",
                        line_color=colors[str(i_ref)],
                        opacity=0.5,
                        row=1,
                        col=col,
                    )

        arrow = "↑" if better == "higher" else "↓"
        fig.update_yaxes(title_text=f"{metric_name} ({arrow} better)", row=1, col=col)
        fig.update_xaxes(title_text=param_name, row=1, col=col)

    fig.update_layout(
        height=420,
        title_text=f"GT Alignment Sensitivity: {param_name} {title_suffix}",
        template="plotly_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    )

    fig.show()
    return fig


def summarize_gt_table(df, param_name):
    """Return compact GT summary table by parameter value and i_ref."""
    cols = ["param_value", "i_ref", "GT RMSE (mV)", "GT MAE (mV)", "GT Valid Intervals"]
    cols = [c for c in cols if c in df.columns]
    out = df[cols].copy().sort_values(["i_ref", "param_value"])
    out.insert(0, "param_name", param_name)
    return out

# Core metric plot
fig_window = plot_sensitivity_results(
    results_window_size,
    "initial_window_size",
    baseline_metrics,
    "(dashed = baseline)",
)

# GT metric plot
fig_window_gt = plot_gt_sensitivity(
    results_window_size,
    "initial_window_size",
    baseline_gt_metrics,
    "(dashed = baseline)",
)

# GT comparison table
gt_table_window = summarize_gt_table(results_window_size, "initial_window_size")
display(gt_table_window.head(15))

if SAVE_PLOTS:
    fig_window.write_html(PLOTS_OUTPUT_DIR / "sensitivity_initial_window_size_core.html")
    fig_window_gt.write_html(PLOTS_OUTPUT_DIR / "sensitivity_initial_window_size_gt.html")
    print("Saved initial_window_size plots.")

,param_name,param_value,i_ref,GT RMSE (mV),GT MAE (mV),GT Valid Intervals
0,initial_window_size,2,0.28,7.939,7.167,400
3,initial_window_size,3,0.28,8.229,7.462,315
6,initial_window_size,4,0.28,8.751,7.985,224
9,initial_window_size,5,0.28,8.524,7.914,169
12,initial_window_size,6,0.28,8.847,8.207,131
1,initial_window_size,2,1.00,14.404,12.800,400
4,initial_window_size,3,1.00,14.057,12.478,315
7,initial_window_size,4,1.00,13.515,12.040,224
10,initial_window_size,5,1.00,13.398,12.089,169
13,initial_window_size,6,1.00,13.216,11.947,131


Saved initial_window_size plots.


---
# Experiment 2: `fixed_freshness_threshold`

`fixed_freshness_threshold` defines how much recent data must be represented in a candidate window.

In [9]:
# =============================================================================
# Cell 9: Run fixed_freshness_threshold experiment
# =============================================================================

results_target_ratio = run_sensitivity_experiment(
    param_name="fixed_freshness_threshold",
    param_values=PARAM_GRIDS["fixed_freshness_threshold"],
    data=data,
    name=extracted_name,
    shared_pre=shared_pre,
    common_config=COMMON_CONFIG,
    adaptive_default=ADAPTIVE_DEFAULT,
    iref_list=IREF_LIST,
    gt_by_iref=GT_BY_IREF,
)

print(f"\nCollected {len(results_target_ratio)} rows")
display(results_target_ratio.head(10))


Testing parameter: fixed_freshness_threshold
Values: [0.2, 0.25, 0.3, 0.35, 0.4]


Testing fixed_freshness_threshold:   0%|          | 0/5 [00:00<?, ?it/s]

Using shared preprocessed data (359353 points, skipping preprocess).
Voltage model fitting ...
Starting Adaptive Fitting for approx 757 intervals...
416 out of 762 fitting results are reliable.
Initialized Urc1_Adaptive (Simplified): Init_Win=3d
  Filters: Target_Min_Ratio=0.1, Freshness=0.2
Using shared preprocessed data (359353 points, skipping preprocess).
Voltage model fitting ...
Starting Adaptive Fitting for approx 757 intervals...
360 out of 762 fitting results are reliable.
Initialized Urc1_Adaptive (Simplified): Init_Win=3d
  Filters: Target_Min_Ratio=0.1, Freshness=0.25
Using shared preprocessed data (359353 points, skipping preprocess).
Voltage model fitting ...
Starting Adaptive Fitting for approx 757 intervals...
315 out of 762 fitting results are reliable.
Initialized Urc1_Adaptive (Simplified): Init_Win=3d
  Filters: Target_Min_Ratio=0.1, Freshness=0.3
Using shared preprocessed data (359353 points, skipping preprocess).
Voltage model fitting ...
Starting Adaptive Fitting

,Target Current (A/cm2),Data Points (n),RMSE (mV),Slope Sigma (uV/h),Mono (Rank) [0-1],Outlier Count,Outlier (%),Max Residual (mV),Mean SE (mV),GT RMSE (mV),GT MAE (mV),GT Valid Intervals,param_name,param_value,i_ref
0,0.28,416,2.813,0.027,0.983,5,1.20,13.663,1.931,8.042,7.349,416,fixed_freshness_threshold,0.20,0.28
1,1.00,416,6.667,0.065,0.983,1,0.24,24.228,1.966,14.519,13.042,416,fixed_freshness_threshold,0.20,1.00
2,1.31,416,9.825,0.096,0.983,1,0.24,33.343,2.271,13.920,10.963,416,fixed_freshness_threshold,0.20,1.31
3,0.28,360,2.909,0.030,0.982,4,1.11,13.520,1.948,8.157,7.426,360,fixed_freshness_threshold,0.25,0.28
4,1.00,360,6.774,0.071,0.982,1,0.28,24.058,1.986,14.340,12.827,360,fixed_freshness_threshold,0.25,1.00
5,1.31,360,9.908,0.104,0.982,1,0.28,33.148,2.307,13.565,10.741,360,fixed_freshness_threshold,0.25,1.31
6,0.28,315,2.952,0.033,0.982,4,1.27,13.313,1.898,8.229,7.462,315,fixed_freshness_threshold,0.30,0.28
7,1.00,315,6.756,0.076,0.984,0,0.00,17.974,1.937,14.057,12.478,315,fixed_freshness_threshold,0.30,1.00
8,1.31,315,9.812,0.110,0.984,0,0.00,26.385,2.255,13.124,10.298,315,fixed_freshness_threshold,0.30,1.31
9,0.28,250,2.958,0.036,0.985,4,1.60,12.847,1.916,8.567,7.779,250,fixed_freshness_threshold,0.35,0.28


In [10]:
# =============================================================================
# Cell 10: Plot fixed_freshness_threshold results + GT table
# =============================================================================

fig_ratio = plot_sensitivity_results(
    results_target_ratio,
    "fixed_freshness_threshold",
    baseline_metrics,
    "(dashed = baseline)",
)

fig_ratio_gt = plot_gt_sensitivity(
    results_target_ratio,
    "fixed_freshness_threshold",
    baseline_gt_metrics,
    "(dashed = baseline)",
)

gt_table_ratio = summarize_gt_table(results_target_ratio, "fixed_freshness_threshold")
display(gt_table_ratio.head(15))

if SAVE_PLOTS:
    fig_ratio.write_html(PLOTS_OUTPUT_DIR / "sensitivity_fixed_freshness_threshold_core.html")
    fig_ratio_gt.write_html(PLOTS_OUTPUT_DIR / "sensitivity_fixed_freshness_threshold_gt.html")
    print("Saved fixed_freshness_threshold plots.")

,param_name,param_value,i_ref,GT RMSE (mV),GT MAE (mV),GT Valid Intervals
0,fixed_freshness_threshold,0.20,0.28,8.042,7.349,416
3,fixed_freshness_threshold,0.25,0.28,8.157,7.426,360
6,fixed_freshness_threshold,0.30,0.28,8.229,7.462,315
9,fixed_freshness_threshold,0.35,0.28,8.567,7.779,250
12,fixed_freshness_threshold,0.40,0.28,8.793,8.017,206
1,fixed_freshness_threshold,0.20,1.00,14.519,13.042,416
4,fixed_freshness_threshold,0.25,1.00,14.340,12.827,360
7,fixed_freshness_threshold,0.30,1.00,14.057,12.478,315
10,fixed_freshness_threshold,0.35,1.00,13.753,12.102,250
13,fixed_freshness_threshold,0.40,1.00,13.555,11.923,206


Saved fixed_freshness_threshold plots.


---
# Experiment 3: `decay_rate`

`decay_rate` controls temporal weighting in WLS:
- `0.0`: equal weighting (close to OLS)
- larger value: stronger emphasis on recent samples

In [11]:
# =============================================================================
# Cell 11: Run decay_rate experiment
# =============================================================================

results_decay = run_sensitivity_experiment(
    param_name="decay_rate",
    param_values=PARAM_GRIDS["decay_rate"],
    data=data,
    name=extracted_name,
    shared_pre=shared_pre,
    common_config=COMMON_CONFIG,
    adaptive_default=ADAPTIVE_DEFAULT,
    iref_list=IREF_LIST,
    gt_by_iref=GT_BY_IREF,
)

print(f"\nCollected {len(results_decay)} rows")
display(results_decay.head(10))


Testing parameter: decay_rate
Values: [0.1, 0.15, 0.2, 0.25, 0.3]


Testing decay_rate:   0%|          | 0/5 [00:00<?, ?it/s]

Using shared preprocessed data (359353 points, skipping preprocess).
Voltage model fitting ...
Starting Adaptive Fitting for approx 757 intervals...
314 out of 762 fitting results are reliable.
Initialized Urc1_Adaptive (Simplified): Init_Win=3d
  Filters: Target_Min_Ratio=0.1, Freshness=0.3
Using shared preprocessed data (359353 points, skipping preprocess).
Voltage model fitting ...
Starting Adaptive Fitting for approx 757 intervals...
315 out of 762 fitting results are reliable.
Initialized Urc1_Adaptive (Simplified): Init_Win=3d
  Filters: Target_Min_Ratio=0.1, Freshness=0.3
Using shared preprocessed data (359353 points, skipping preprocess).
Voltage model fitting ...
Starting Adaptive Fitting for approx 757 intervals...
315 out of 762 fitting results are reliable.
Initialized Urc1_Adaptive (Simplified): Init_Win=3d
  Filters: Target_Min_Ratio=0.1, Freshness=0.3
Using shared preprocessed data (359353 points, skipping preprocess).
Voltage model fitting ...
Starting Adaptive Fitting 

,Target Current (A/cm2),Data Points (n),RMSE (mV),Slope Sigma (uV/h),Mono (Rank) [0-1],Outlier Count,Outlier (%),Max Residual (mV),Mean SE (mV),GT RMSE (mV),GT MAE (mV),GT Valid Intervals,param_name,param_value,i_ref
0,0.28,314,2.856,0.032,0.982,3,0.96,10.259,1.891,8.128,7.412,314,decay_rate,0.10,0.28
1,1.00,314,6.750,0.076,0.984,0,0.00,18.024,1.931,14.049,12.471,314,decay_rate,0.10,1.00
2,1.31,314,9.847,0.111,0.984,0,0.00,26.588,2.241,13.155,10.328,314,decay_rate,0.10,1.31
3,0.28,315,2.963,0.033,0.982,4,1.27,14.164,1.895,8.238,7.467,315,decay_rate,0.15,0.28
4,1.00,315,6.748,0.076,0.984,0,0.00,17.996,1.935,14.048,12.472,315,decay_rate,0.15,1.00
5,1.31,315,9.822,0.110,0.984,0,0.00,26.524,2.252,13.130,10.308,315,decay_rate,0.15,1.31
6,0.28,315,2.952,0.033,0.982,4,1.27,13.313,1.898,8.229,7.462,315,decay_rate,0.20,0.28
7,1.00,315,6.756,0.076,0.984,0,0.00,17.974,1.937,14.057,12.478,315,decay_rate,0.20,1.00
8,1.31,315,9.812,0.110,0.984,0,0.00,26.385,2.255,13.124,10.298,315,decay_rate,0.20,1.31
9,0.28,315,2.944,0.033,0.982,4,1.27,12.530,1.901,8.221,7.457,315,decay_rate,0.25,0.28


In [12]:
# =============================================================================
# Cell 12: Plot decay_rate results + GT table
# =============================================================================

fig_decay = plot_sensitivity_results(
    results_decay,
    "decay_rate",
    baseline_metrics,
    "(dashed = baseline)",
)

fig_decay_gt = plot_gt_sensitivity(
    results_decay,
    "decay_rate",
    baseline_gt_metrics,
    "(dashed = baseline)",
)

gt_table_decay = summarize_gt_table(results_decay, "decay_rate")
display(gt_table_decay.head(15))

if SAVE_PLOTS:
    fig_decay.write_html(PLOTS_OUTPUT_DIR / "sensitivity_decay_rate_core.html")
    fig_decay_gt.write_html(PLOTS_OUTPUT_DIR / "sensitivity_decay_rate_gt.html")
    print("Saved decay_rate plots.")

,param_name,param_value,i_ref,GT RMSE (mV),GT MAE (mV),GT Valid Intervals
0,decay_rate,0.10,0.28,8.128,7.412,314
3,decay_rate,0.15,0.28,8.238,7.467,315
6,decay_rate,0.20,0.28,8.229,7.462,315
9,decay_rate,0.25,0.28,8.221,7.457,315
12,decay_rate,0.30,0.28,8.214,7.452,315
1,decay_rate,0.10,1.00,14.049,12.471,314
4,decay_rate,0.15,1.00,14.048,12.472,315
7,decay_rate,0.20,1.00,14.057,12.478,315
10,decay_rate,0.25,1.00,14.066,12.485,315
13,decay_rate,0.30,1.00,14.075,12.491,315


Saved decay_rate plots.


---
# Global Sensitivity Summary

Compute sensitivity indices and include GT-oriented ranking metrics.

In [13]:
# =============================================================================
# Cell 13: Sensitivity index (including GT metrics)
# =============================================================================

def calculate_sensitivity_index(results_dict):
    """Compute coefficient of variation (CV%) per metric and parameter."""
    metrics_cols = [
        "Data Points (n)",
        "RMSE (mV)",
        "Mono (Rank) [0-1]",
        "Slope Sigma (uV/h)",
        "Outlier (%)",
        "GT RMSE (mV)",
        "GT MAE (mV)",
        "GT Valid Intervals",
    ]

    sensitivity_rows = []

    for param_name, df in results_dict.items():
        for metric in metrics_cols:
            if metric not in df.columns:
                continue

            grouped = df.groupby("param_value")[metric].mean()
            grouped = grouped.replace([np.inf, -np.inf], np.nan).dropna()
            if grouped.empty:
                continue

            mean_val = grouped.mean()
            cv = (grouped.std() / mean_val) * 100 if mean_val != 0 else 0

            sensitivity_rows.append(
                {
                    "Parameter": param_name,
                    "Metric": metric,
                    "CV (%)": round(cv, 2),
                    "Min": round(grouped.min(), 3),
                    "Max": round(grouped.max(), 3),
                    "Range": round(grouped.max() - grouped.min(), 3),
                }
            )

    return pd.DataFrame(sensitivity_rows)

all_results = {
    "initial_window_size": results_window_size,
    "fixed_freshness_threshold": results_target_ratio,
    "decay_rate": results_decay,
}

sensitivity_df = calculate_sensitivity_index(all_results)
print("\n" + "=" * 70)
print("Sensitivity index table (CV%: larger means more sensitive)")
print("=" * 70)
display(sensitivity_df)


Sensitivity index table (CV%: larger means more sensitive)


,Parameter,Metric,CV (%),Min,Max,Range
0,initial_window_size,Data Points (n),44.25,131.000,400.000,269.000
1,initial_window_size,RMSE (mV),7.66,5.700,6.782,1.082
2,initial_window_size,Mono (Rank) [0-1],0.17,0.981,0.986,0.005
3,initial_window_size,Slope Sigma (uV/h),15.87,0.067,0.100,0.033
4,initial_window_size,Outlier (%),45.63,0.197,0.833,0.637
5,initial_window_size,GT RMSE (mV),4.22,10.989,12.079,1.090
6,initial_window_size,GT MAE (mV),3.41,9.567,10.311,0.744
7,initial_window_size,GT Valid Intervals,44.25,131.000,400.000,269.000
8,fixed_freshness_threshold,Data Points (n),27.13,206.000,416.000,210.000
9,fixed_freshness_threshold,RMSE (mV),0.85,6.435,6.577,0.142


In [14]:
# =============================================================================
# Cell 14: Sensitivity heatmap
# =============================================================================

pivot_cv = sensitivity_df.pivot(index="Parameter", columns="Metric", values="CV (%)")

fig_heatmap = px.imshow(
    pivot_cv,
    labels=dict(x="Metric", y="Parameter", color="CV (%)"),
    title="Parameter Sensitivity Heatmap (CV%)",
    color_continuous_scale="RdYlBu_r",
    aspect="auto",
    text_auto=True,
)

fig_heatmap.update_layout(height=420, template="plotly_white")
fig_heatmap.show()

if SAVE_PLOTS:
    fig_heatmap.write_html(PLOTS_OUTPUT_DIR / "sensitivity_heatmap_cv.html")
    print("Saved sensitivity heatmap.")

Saved sensitivity heatmap.


In [15]:
# =============================================================================
# Cell 15: Sensitivity ranking
# =============================================================================

avg_sensitivity = (
    sensitivity_df.groupby("Parameter")["CV (%)"]
    .mean()
    .sort_values(ascending=False)
)

print("\n" + "=" * 70)
print("Parameter sensitivity ranking (by mean CV%)")
print("=" * 70)
for i, (param, cv) in enumerate(avg_sensitivity.items(), 1):
    print(f"  {i}. {param}: {cv:.2f}%")

# GT-specific ranking (focus on GT RMSE CV)
gt_focus = sensitivity_df[sensitivity_df["Metric"] == "GT RMSE (mV)"]
if not gt_focus.empty:
    gt_rank = gt_focus.sort_values("CV (%)", ascending=False)
    print("\nGT-focused ranking (GT RMSE CV%):")
    for i, (_, row) in enumerate(gt_rank.iterrows(), 1):
        print(f"  {i}. {row['Parameter']}: {row['CV (%)']:.2f}%")


Parameter sensitivity ranking (by mean CV%)
  1. initial_window_size: 20.68%
  2. fixed_freshness_threshold: 11.83%
  3. decay_rate: 2.26%

GT-focused ranking (GT RMSE CV%):
  1. initial_window_size: 4.22%
  2. fixed_freshness_threshold: 2.27%
  3. decay_rate: 0.10%


In [16]:
# =============================================================================
# Cell 16: Save results
# =============================================================================

all_experiments = pd.concat(
    [results_window_size, results_target_ratio, results_decay],
    ignore_index=True,
)

output_path = f"sensitivity_analysis_{extracted_name}.csv"
all_experiments.to_csv(output_path, index=False)
print(f"Results saved: {output_path}")

sensitivity_path = f"sensitivity_index_{extracted_name}.csv"
sensitivity_df.to_csv(sensitivity_path, index=False)
print(f"Sensitivity index saved: {sensitivity_path}")

# Save GT-only summary table
gt_cols = [
    "param_name",
    "param_value",
    "i_ref",
    "GT RMSE (mV)",
    "GT MAE (mV)",
    "GT Valid Intervals",
]
gt_cols = [c for c in gt_cols if c in all_experiments.columns]
if gt_cols:
    gt_summary_path = f"sensitivity_gt_summary_{extracted_name}.csv"
    all_experiments[gt_cols].to_csv(gt_summary_path, index=False)
    print(f"GT summary saved: {gt_summary_path}")

if SAVE_PLOTS:
    print(f"Plot files saved under: {PLOTS_OUTPUT_DIR.resolve()}")

Results saved: sensitivity_analysis_G6M2.csv
Sensitivity index saved: sensitivity_index_G6M2.csv
GT summary saved: sensitivity_gt_summary_G6M2.csv
Plot files saved under: C:\Users\Z0057NPT\Documents\MA_code\master_arbeit_Di\plots\ada\G6M2_sensitivity
